In [23]:
%pwd

'/kaggle/working'

In [24]:
!pip install ultralytics torch torchvision torchaudio scikit-learn pandas matplotlib seaborn pillow numpy

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [25]:
!python -c "import torch; print(f'PyTorch Version: {torch.__version__}'); print(f'CUDA Available: {torch.cuda.is_available()}')"

PyTorch Version: 2.10.0+cu128
CUDA Available: True


In [26]:
%pwd

'/kaggle/working'

In [27]:
%cd ..

/kaggle


In [28]:
%ls

input/  lib/  working/


In [29]:
%cd input/

/kaggle/input


In [30]:
%ls

datasets/


In [31]:
%cd datasets/

/kaggle/input/datasets


In [32]:
%ls

gudhijagadeeshkumar/


In [33]:
%cd gudhijagadeeshkumar

/kaggle/input/datasets/gudhijagadeeshkumar


In [34]:
%ls

urban-asset-detection-dataset/


In [35]:
%cd urban-asset-detection-dataset

/kaggle/input/datasets/gudhijagadeeshkumar/urban-asset-detection-dataset


In [36]:
%ls

yolo_dataset/


In [37]:
%cd yolo_dataset

/kaggle/input/datasets/gudhijagadeeshkumar/urban-asset-detection-dataset/yolo_dataset


In [38]:
%ls

data.yaml  images/  labels/


In [39]:
%cd /kaggle/working

/kaggle/working


In [40]:
%ls

In [48]:
%ls

In [88]:
#!/usr/bin/env python3
"""
Fix satellite image filenames to match their label files.
YOLO matches images to labels by filename stem (without extension).
Current issue: Images are named with _sat suffix but labels are not.

This script renames all _sat.jpg files to remove the suffix.
"""

import os
from pathlib import Path


def fix_satellite_filenames(dataset_root, splits=['train', 'val', 'test']):
    """Rename satellite images to remove _sat suffix."""
    
    dataset_root = Path(dataset_root)
    total_renamed = 0
    
    for split in splits:
        images_dir = dataset_root / 'yolo_dataset' / 'images' / split
        labels_dir = dataset_root / 'yolo_dataset' / 'labels' / split
        
        if not images_dir.exists():
            print(f"⚠️  {split} images directory not found: {images_dir}")
            continue
        
        # Find all satellite images
        sat_images = list(images_dir.glob('*_sat.jpg'))
        
        if not sat_images:
            print(f"✓ {split}: No satellite images found")
            continue
        
        print(f"\n{split.upper()} split:")
        print(f"  Found {len(sat_images)} satellite images")
        
        # Check which ones will be renamed
        to_rename = []
        for sat_img in sat_images:
            # Get the stem without _sat
            new_stem = sat_img.stem.replace('_sat', '')
            new_name = f"{new_stem}.jpg"
            
            # Check if label exists
            label_file = labels_dir / f"{new_stem}.txt"
            if label_file.exists():
                to_rename.append((sat_img, sat_img.parent / new_name, new_stem))
            else:
                print(f"  ⚠️  Missing label for {sat_img.name}: {new_stem}.txt")
        
        # Rename the files
        if to_rename:
            print(f"  Renaming {len(to_rename)} images...")
            for old_path, new_path, stem in to_rename:
                old_path.rename(new_path)
                print(f"    ✓ {old_path.name} → {new_path.name}")
                total_renamed += 1
        
        # Verify all images now have labels
        print(f"  Validating...")
        orphaned = 0
        for img_path in images_dir.glob('*.jpg'):
            stem = img_path.stem
            if not (labels_dir / f"{stem}.txt").exists():
                print(f"    ⚠️  No label for {img_path.name}")
                orphaned += 1
        
        if orphaned == 0:
            print(f"  ✅ All {len(list(images_dir.glob('*.jpg')))} images have labels")
        else:
            print(f"  ⚠️  {orphaned} images missing labels")
    
    print(f"\n{'='*50}")
    print(f"✅ Total images renamed: {total_renamed}")
    print(f"Dataset is now ready for training with proper filename matching!")

In [90]:
!cp -r "/kaggle/input/datasets/gudhijagadeeshkumar/urban-asset-detection-dataset/" "/kaggle/working"

In [91]:
dataset_root = "/kaggle/working/urban-asset-detection-dataset/"
fix_satellite_filenames(dataset_root)


TRAIN split:
  Found 1421 satellite images
  Renaming 1421 images...
    ✓ 518439_sat.jpg → 518439.jpg
    ✓ 48710_sat.jpg → 48710.jpg
    ✓ 935147_sat.jpg → 935147.jpg
    ✓ 411059_sat.jpg → 411059.jpg
    ✓ 311386_sat.jpg → 311386.jpg
    ✓ 410319_sat.jpg → 410319.jpg
    ✓ 862095_sat.jpg → 862095.jpg
    ✓ 243913_sat.jpg → 243913.jpg
    ✓ 405830_sat.jpg → 405830.jpg
    ✓ 166293_sat.jpg → 166293.jpg
    ✓ 291214_sat.jpg → 291214.jpg
    ✓ 538243_sat.jpg → 538243.jpg
    ✓ 333000_sat.jpg → 333000.jpg
    ✓ 914008_sat.jpg → 914008.jpg
    ✓ 200602_sat.jpg → 200602.jpg
    ✓ 772130_sat.jpg → 772130.jpg
    ✓ 646979_sat.jpg → 646979.jpg
    ✓ 159882_sat.jpg → 159882.jpg
    ✓ 643780_sat.jpg → 643780.jpg
    ✓ 37755_sat.jpg → 37755.jpg
    ✓ 185522_sat.jpg → 185522.jpg
    ✓ 333857_sat.jpg → 333857.jpg
    ✓ 291871_sat.jpg → 291871.jpg
    ✓ 555777_sat.jpg → 555777.jpg
    ✓ 671952_sat.jpg → 671952.jpg
    ✓ 491696_sat.jpg → 491696.jpg
    ✓ 702918_sat.jpg → 702918.jpg
    ✓ 28559_sat.

In [50]:
import os
import json
import torch
import argparse
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from PIL import Image

from ultralytics import YOLO
from sklearn.metrics import confusion_matrix, classification_report

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [92]:
"""
Complete YOLO training pipeline with GPU support and comprehensive metrics.

Features:
- GPU-accelerated training with YOLOv8
- Real-time training/validation loss plotting
- Confusion matrix visualization
- Test set evaluation and metrics
- Training report generation
- Model comparison and recommendations

Usage:
    python train_yolo_model.py \
        --data yolo_dataset/data.yaml \
        --model yolov8m.pt \
        --epochs 100 \
        --batch 32 \
        --device 0
"""

import os
import json
import torch
import argparse
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from PIL import Image

from ultralytics import YOLO
from sklearn.metrics import confusion_matrix, classification_report


class YOLOTrainer:
    """Complete YOLO training pipeline with metrics"""
    
    def __init__(self, data_yaml, model_name='yolov8m', device=0):
        self.data_yaml = Path(data_yaml)
        self.model_name = model_name
        self.device = device
        self.results = None
        self.model = None
        self.metrics = {}
        
        # Create output directory
        self.output_dir = Path('training_results') / datetime.now().strftime('%Y%m%d_%H%M%S')
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"\n{'=' * 70}")
        print("YOLO Training Pipeline")
        print(f"{'=' * 70}")
        print(f"Data YAML: {self.data_yaml}")
        print(f"Model: {model_name}")
        print(f"Device: {device} (GPU: {torch.cuda.get_device_name(device) if torch.cuda.is_available() else 'N/A'})")
        print(f"Output dir: {self.output_dir}")
        print(f"{'=' * 70}")
    
    def train(self, epochs=100, batch_size=32, img_size=640, patience=20):
        """Train YOLO model"""
        print(f"\n{'=' * 70}")
        print("STEP 1: Training")
        print(f"{'=' * 70}")
        
        # Load model
        self.model = YOLO(f'{self.model_name}.pt')
        
        # Train
        self.results = self.model.train(
            data=str(self.data_yaml),
            epochs=epochs,
            imgsz=img_size,
            batch=batch_size,
            device=self.device,
            patience=patience,
            verbose=True,
            save=True,
            project=str(self.output_dir.parent),
            name=self.output_dir.name
        )
        
        print(f"\n[OK] Training complete!")
        self.model = YOLO(self.output_dir.parent / self.output_dir.name / 'weights' / 'best.pt')
    
    def evaluate_test_set(self, data_yaml=None):
        """Evaluate on test set"""
        print(f"\n{'=' * 70}")
        print("STEP 2: Test Set Evaluation")
        print(f"{'=' * 70}")
        
        if data_yaml is None:
            data_yaml = self.data_yaml
        
        metrics = self.model.val(data=str(data_yaml), device=self.device)
        
        self.metrics['test'] = {
            'mAP50': metrics.box.map50,
            'mAP50-95': metrics.box.map,
            'precision': metrics.box.mp,
            'recall': metrics.box.mr,
        }
        
        print(f"\nTest Set Metrics:")
        print(f"  mAP50:     {self.metrics['test']['mAP50']:.4f}")
        print(f"  mAP50-95:  {self.metrics['test']['mAP50-95']:.4f}")
        print(f"  Precision: {self.metrics['test']['precision']:.4f}")
        print(f"  Recall:    {self.metrics['test']['recall']:.4f}")
        
        return metrics
    
    def generate_confusion_matrix(self):
        """Generate confusion matrix using YOLO's built-in functionality"""
        print(f"\n{'=' * 70}")
        print("STEP 3: Confusion Matrix Generation")
        print(f"{'=' * 70}")
        
        try:
            # Use YOLO's built-in confusion matrix generation
            print("Generating confusion matrix from validation results...")
            metrics = self.model.val(
                data=str(self.data_yaml),
                device=self.device,
                plots=True,
                save_json=False,
                verbose=False
            )
            
            print(f"[OK] Confusion matrix generated by YOLO")
            
            # Try to copy YOLO's confusion matrix to our output dir
            try:
                import shutil
                yolo_cm_path = list(Path('runs/detect').glob('*/confusion_matrix.png'))
                if yolo_cm_path:
                    shutil.copy(str(yolo_cm_path[-1]), self.output_dir / 'confusion_matrix.png')
                    print(f"[OK] Confusion matrix saved to {self.output_dir / 'confusion_matrix.png'}")
            except Exception as e:
                print(f"Note: Could not copy confusion matrix: {e}")
            
            return metrics
        
        except Exception as e:
            print(f"⚠ Error generating confusion matrix: {e}")
            print(f"Note: YOLO's built-in confusion matrix uses proper IoU matching")
            return None
    
    def plot_training_metrics(self):
        """Plot training/validation losses and metrics"""
        print(f"\n{'=' * 70}")
        print("STEP 4: Training Metrics Visualization")
        print(f"{'=' * 70}")
        
        # Read results from CSV
        csv_path = self.output_dir / 'results.csv'
        if not csv_path.exists():
            print(f"⚠ Results CSV not found at {csv_path}")
            return
        
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        
        # Create comprehensive plot
        fig = plt.figure(figsize=(16, 12))
        gs = GridSpec(3, 2, figure=fig, hspace=0.3, wspace=0.3)
        
        # Loss plot
        ax1 = fig.add_subplot(gs[0, :])
        if 'train/loss' in df.columns and 'val/loss' in df.columns:
            ax1.plot(df.index, df['train/loss'], label='Train Loss', linewidth=2, marker='o', markersize=3)
            ax1.plot(df.index, df['val/loss'], label='Val Loss', linewidth=2, marker='s', markersize=3)
            ax1.set_xlabel('Epoch', fontsize=11, fontweight='bold')
            ax1.set_ylabel('Loss', fontsize=11, fontweight='bold')
            ax1.set_title('Training vs Validation Loss', fontsize=12, fontweight='bold')
            ax1.legend(fontsize=10)
            ax1.grid(True, alpha=0.3)
        
        # Box loss
        ax2 = fig.add_subplot(gs[1, 0])
        if 'train/box_loss' in df.columns and 'val/box_loss' in df.columns:
            ax2.plot(df.index, df['train/box_loss'], label='Train', linewidth=2, marker='o', markersize=3, color='#1f77b4')
            ax2.plot(df.index, df['val/box_loss'], label='Val', linewidth=2, marker='s', markersize=3, color='#ff7f0e')
            ax2.set_xlabel('Epoch', fontsize=10, fontweight='bold')
            ax2.set_ylabel('Box Loss', fontsize=10, fontweight='bold')
            ax2.set_title('Box Loss', fontsize=11, fontweight='bold')
            ax2.legend(fontsize=9)
            ax2.grid(True, alpha=0.3)
        
        # Class loss
        ax3 = fig.add_subplot(gs[1, 1])
        if 'train/cls_loss' in df.columns and 'val/cls_loss' in df.columns:
            ax3.plot(df.index, df['train/cls_loss'], label='Train', linewidth=2, marker='o', markersize=3, color='#2ca02c')
            ax3.plot(df.index, df['val/cls_loss'], label='Val', linewidth=2, marker='s', markersize=3, color='#d62728')
            ax3.set_xlabel('Epoch', fontsize=10, fontweight='bold')
            ax3.set_ylabel('Class Loss', fontsize=10, fontweight='bold')
            ax3.set_title('Class Loss', fontsize=11, fontweight='bold')
            ax3.legend(fontsize=9)
            ax3.grid(True, alpha=0.3)
        
        # mAP
        ax4 = fig.add_subplot(gs[2, 0])
        if 'metrics/mAP50' in df.columns:
            ax4.plot(df.index, df['metrics/mAP50'], linewidth=2, marker='o', markersize=4, color='#9467bd')
            ax4.set_xlabel('Epoch', fontsize=10, fontweight='bold')
            ax4.set_ylabel('mAP50', fontsize=10, fontweight='bold')
            ax4.set_title('Mean Average Precision (mAP50)', fontsize=11, fontweight='bold')
            ax4.grid(True, alpha=0.3)
            ax4.set_ylim([0, 1])
        
        # Metrics distribution
        ax5 = fig.add_subplot(gs[2, 1])
        if 'metrics/mAP50' in df.columns:
            metrics_data = {
                'mAP50': df['metrics/mAP50'].iloc[-1] if 'metrics/mAP50' in df.columns else 0,
                'mAP50-95': df['metrics/mAP'] .iloc[-1] if 'metrics/mAP' in df.columns else 0,
            }
            colors_bar = ['#1f77b4', '#ff7f0e']
            bars = ax5.bar(metrics_data.keys(), metrics_data.values(), color=colors_bar, alpha=0.8, edgecolor='black', linewidth=1.5)
            ax5.set_ylabel('Score', fontsize=10, fontweight='bold')
            ax5.set_title('Final Test Metrics', fontsize=11, fontweight='bold')
            ax5.set_ylim([0, 1])
            for bar in bars:
                height = bar.get_height()
                ax5.text(bar.get_x() + bar.get_width()/2., height,
                        f'{height:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        plt.suptitle('YOLO Training Metrics Dashboard', fontsize=16, fontweight='bold', y=0.995)
        plt.savefig(self.output_dir / 'training_metrics.png', dpi=300, bbox_inches='tight')
        print(f"[OK] Training metrics plot saved")
        plt.close()
    
    def generate_report(self, test_metrics=None):
        """Generate comprehensive training report"""
        print(f"\n{'=' * 70}")
        print("STEP 5: Report Generation")
        print(f"{'=' * 70}")
        
        report_path = self.output_dir / 'TRAINING_REPORT.md'
        
        with open(report_path, 'w') as f:
            f.write("# YOLO Model Training Report\n\n")
            f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            # Configuration
            f.write("## Training Configuration\n\n")
            f.write(f"- **Model:** {self.model_name}\n")
            f.write(f"- **Device:** GPU {self.device}\n")
            f.write(f"- **Data YAML:** {self.data_yaml}\n")
            f.write(f"- **Output Directory:** {self.output_dir}\n\n")
            
            # Test Metrics
            if test_metrics:
                f.write("## Test Set Metrics\n\n")
                f.write("| Metric | Score |\n")
                f.write("|--------|-------|\n")
                for key, value in self.metrics.get('test', {}).items():
                    f.write(f"| {key} | {value:.4f} |\n")
                f.write("\n")
            
            # Class Distribution
            f.write("## Class Distribution\n\n")
            f.write("- **Building:** 17.3%\n")
            f.write("- **Road:** 2.7%\n")
            f.write("- **Water Body:** 2.7%\n")
            f.write("- **Vegetation:** 11.6%\n")
            f.write("- **Vehicle:** 65.6%\n\n")
            
            # Dataset
            f.write("## Dataset Information\n\n")
            f.write("| Split | Images | Labels |\n")
            f.write("|-------|--------|--------|\n")
            f.write("| Train | 3,648  | 3,648  |\n")
            f.write("| Val   | 456    | 456    |\n")
            f.write("| Test  | 456    | 456    |\n")
            f.write("| Total | 4,560  | 4,560  |\n\n")
            
            # Performance Insights
            f.write("## Performance Insights\n\n")
            f.write("### Strengths:\n")
            f.write("- High vehicle detection capability (class imbalance advantage)\n")
            f.write("- Multi-class object detection across 5 categories\n")
            f.write("- Balanced train/val/test splits\n\n")
            
            f.write("### Recommendations:\n")
            f.write("- Consider class weighting to improve minority class detection\n")
            f.write("- Augmentation strategies for road and water_body classes\n")
            f.write("- Use ensemble methods for production deployment\n\n")
            
            # Output files
            f.write("## Generated Files\n\n")
            f.write("- `training_metrics.png` - Training/validation loss and metric plots\n")
            f.write("- `confusion_matrix.png` - Confusion matrix visualization\n")
            f.write("- `model/weights/best.pt` - Best trained model\n\n")
        
        print(f"[OK] Report saved to {report_path}")
    
    def create_summary_table(self):
        """Create summary statistics table"""
        print(f"\n{'=' * 70}")
        print("Summary Statistics")
        print(f"{'=' * 70}\n")
        
        summary_data = {
            'Dataset': ['Train', 'Validation', 'Test', 'Total'],
            'Images': [3648, 456, 456, 4560],
            'Labels': [3648, 456, 456, 4560],
            'Split %': ['80.0%', '10.0%', '10.0%', '100%']
        }
        
        df_summary = pd.DataFrame(summary_data)
        print(df_summary.to_string(index=False))
        print()
        
        # Save to CSV
        csv_path = self.output_dir / 'summary_statistics.csv'
        df_summary.to_csv(csv_path, index=False)
        print(f"[OK] Summary saved to {csv_path}\n")


In [52]:
!yolo settings runs_dir=/kaggle/working

✅ Updated 'runs_dir=/kaggle/working'
JSONDict("/root/.config/Ultralytics/settings.json"):
{
  "settings_version": "0.0.6",
  "datasets_dir": "/kaggle/working/datasets",
  "weights_dir": "weights",
  "runs_dir": "/kaggle/working",
  "uuid": "1bfc3e992d24318da58ddee183be5bf9388a31f26bab1738e986ec4d297417ff",
  "sync": true,
  "api_key": "",
  "openai_api_key": "",
  "clearml": true,
  "comet": true,
  "dvc": true,
  "hub": true,
  "mlflow": true,
  "neptune": true,
  "raytune": true,
  "tensorboard": false,
  "wandb": false,
  "vscode_msg": true,
  "openvino_msg": true
}
💡 Learn more about Ultralytics Settings at https://docs.ultralytics.com/quickstart/#ultralytics-settings


In [93]:
!cp /kaggle/input/datasets/gudhijagadeeshkumar/urban-asset-detection-dataset/yolo_dataset/data.yaml /kaggle/working/data.yaml

In [95]:
!sed -i 's|^path:.*|path: /kaggle/working/urban-asset-detection-dataset/yolo_dataset|' data.yaml

In [96]:
%cat data.yaml

path: /kaggle/working/urban-asset-detection-dataset/yolo_dataset
train: images/train
val: images/val
test: images/test

nc: 5
names: ['building', 'road', 'water_body', 'vegetation', 'vehicle']


In [97]:
data = "/kaggle/working/data.yaml"
model_name = "yolov8m"
epochs = 40
batch = 8
device = 0
image_size = 640
patience = 20

In [98]:
trainer = YOLOTrainer(data, model_name, device)


YOLO Training Pipeline
Data YAML: /kaggle/working/data.yaml
Model: yolov8m
Device: 0 (GPU: Tesla T4)
Output dir: training_results/20260411_031022


In [99]:
trainer.train(epochs=epochs, batch_size=batch, img_size=image_size, patience=patience)


STEP 1: Training
Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=20260411_031022, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

FileNotFoundError: [Errno 2] No such file or directory: 'training_results/20260411_031022/weights/best.pt'

In [101]:
from IPython.display import FileLink
!zip -r training_results.zip /kaggle/working/runs/detect/training_results/20260411_031022

  adding: kaggle/working/runs/detect/training_results/20260411_031022/ (stored 0%)
  adding: kaggle/working/runs/detect/training_results/20260411_031022/val_batch0_labels.jpg (deflated 7%)
  adding: kaggle/working/runs/detect/training_results/20260411_031022/val_batch1_labels.jpg (deflated 5%)
  adding: kaggle/working/runs/detect/training_results/20260411_031022/train_batch1.jpg (deflated 4%)
  adding: kaggle/working/runs/detect/training_results/20260411_031022/val_batch2_labels.jpg (deflated 5%)
  adding: kaggle/working/runs/detect/training_results/20260411_031022/train_batch0.jpg (deflated 5%)
  adding: kaggle/working/runs/detect/training_results/20260411_031022/results.csv (deflated 61%)
  adding: kaggle/working/runs/detect/training_results/20260411_031022/val_batch2_pred.jpg (deflated 5%)
  adding: kaggle/working/runs/detect/training_results/20260411_031022/BoxF1_curve.png (deflated 8%)
  adding: kaggle/working/runs/detect/training_results/20260411_031022/val_batch1_pred.jpg (defla

In [102]:
FileLink('training_results.zip') 

/kaggle/working/training_results.zip

In [103]:
!zip -r training_evaluations.zip /kaggle/working/training_results/20260411_031022

  adding: kaggle/working/training_results/20260411_031022/ (stored 0%)


In [104]:
FileLink('training_evaluations.zip') 

/kaggle/working/training_evaluations.zip

In [105]:
trainer.evaluate_test_set()


STEP 2: Test Set Evaluation
Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,842,655 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1673.4±1490.2 MB/s, size: 9100.1 KB)
val: Scanning /kaggle/working/urban-asset-detection-dataset/yolo_dataset/labels/val.cache... 456 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 456/456 76.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 1.2it/s 24.1s0.6s
                   all        456      10324       0.63      0.431      0.446      0.294
              building        194       1706      0.858      0.859      0.901      0.642
                  road         99        275      0.599      0.359      0.368      0.255
            water_body         52        332      0.516      0.343      0.358      0.224
            vegetation         88       1151      0.466    

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7dde4136d160>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [106]:
trainer.generate_confusion_matrix()


STEP 3: Confusion Matrix Generation
Generating confusion matrix from validation results...
Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1508.3±1474.6 MB/s, size: 589.9 KB)
val: Scanning /kaggle/working/urban-asset-detection-dataset/yolo_dataset/labels/val.cache... 456 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 456/456 191.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 1.3it/s 21.6s0.5s
                   all        456      10324       0.63      0.431      0.446      0.294
Speed: 1.4ms preprocess, 21.3ms inference, 0.0ms loss, 5.5ms postprocess per image
Results saved to /kaggle/working/runs/detect/val2
[OK] Confusion matrix generated by YOLO
[OK] Confusion matrix saved to training_results/20260411_031022/confusion_matrix.png


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ddde153f920>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [107]:
trainer.plot_training_metrics()


STEP 4: Training Metrics Visualization
⚠ Results CSV not found at training_results/20260411_031022/results.csv


In [108]:
trainer.generate_report()


STEP 5: Report Generation
[OK] Report saved to training_results/20260411_031022/TRAINING_REPORT.md


In [109]:
FileLink("training_results/20260411_031022/TRAINING_REPORT.md")

/kaggle/working/training_results/20260411_031022/TRAINING_REPORT.md